# 04 - Multi-Backend Execution with QONTOS

QONTOS can schedule quantum circuit execution across **multiple backends**:
local simulators, IBM Quantum, Amazon Braket, and (in the future) native
QONTOS modular hardware. The scheduler selects the best backend for each
partition based on configurable policies.

This notebook demonstrates:

1. Listing available backends
2. Configuring scheduling policies
3. Submitting a circuit with backend preferences
4. Comparing results across backends
5. Understanding fidelity-first vs cost-aware scheduling

## Environment

This notebook works with the QONTOS local simulator. No quantum hardware
access is needed. Backend listing uses the SDK's registry, which includes
simulator backends by default.

```bash
# pip install git+https://github.com/qontos/qontos.git@v0.2.0 git+https://github.com/qontos/qontos-sim.git@v0.1.0
```

## Prerequisites

- Python >= 3.10
- `qontos` and `qontos-sim` installed
- Familiarity with [03_partitioning](03_partitioning.ipynb)

In [ ]:
import os

from qontos import QontosClient
from qontos.circuit import CircuitNormalizer
from qontos.scheduling import SchedulingPolicy

## Connect to QONTOS

We connect to a local QONTOS instance. In production, you would point
to the hosted QONTOS endpoint.

In [ ]:
client = QontosClient(
    base_url=os.getenv("QONTOS_BASE_URL", "http://localhost:8000"),
    api_key=os.getenv("QONTOS_API_KEY", "demo"),
)

## List Available Backends

The QONTOS backend registry tracks all configured execution targets.
Each backend reports its capabilities: max qubits, supported gate sets,
whether it is synchronous, and estimated fidelity characteristics.

In [ ]:
# List all backends registered with this QONTOS instance
backends = client.list_backends()

print(f"{'Backend':<25} {'Provider':<15} {'Max Qubits':>10} {'Sync':>6}")
print("-" * 60)
for b in backends:
    print(f"{b.name:<25} {b.provider:<15} {b.max_qubits:>10} {str(b.is_synchronous):>6}")

## Configure Scheduling Policy

QONTOS supports several scheduling policies that control how partitions
are assigned to backends:

| Policy | Description |
|--------|-------------|
| `FIDELITY_FIRST` | Prefer backends with highest estimated fidelity |
| `COST_AWARE` | Balance fidelity against execution cost |
| `LATENCY_FIRST` | Prefer backends with lowest queue time |
| `ROUND_ROBIN` | Distribute evenly across available backends |

The default is `FIDELITY_FIRST`.

In [ ]:
# Define a Bell state circuit
bell_qasm = """\
OPENQASM 2.0;
include "qelib1.inc";
qreg q[2];
creg c[2];
h q[0];
cx q[0],q[1];
measure q[0] -> c[0];
measure q[1] -> c[1];
"""

# Submit with FIDELITY_FIRST scheduling
job_fidelity = client.submit_job(
    circuit=bell_qasm,
    shots=4096,
    name="Multi-Backend: Fidelity First",
    scheduling_policy=SchedulingPolicy.FIDELITY_FIRST,
    tags={"source": "notebook-04", "policy": "fidelity_first"},
)
print(f"Job submitted: {job_fidelity.id}")
print(f"Policy: FIDELITY_FIRST")

In [ ]:
# Submit the same circuit with COST_AWARE scheduling
job_cost = client.submit_job(
    circuit=bell_qasm,
    shots=4096,
    name="Multi-Backend: Cost Aware",
    scheduling_policy=SchedulingPolicy.COST_AWARE,
    tags={"source": "notebook-04", "policy": "cost_aware"},
)
print(f"Job submitted: {job_cost.id}")
print(f"Policy: COST_AWARE")

## Compare Results Across Backends

After both jobs complete, we can compare which backend each policy
selected and how the results differ.

In [ ]:
# Wait for both jobs
job_fidelity = client.wait_for_job(job_fidelity.id, timeout=120)
job_cost = client.wait_for_job(job_cost.id, timeout=120)

# Retrieve results
for label, job in [("Fidelity-First", job_fidelity), ("Cost-Aware", job_cost)]:
    print(f"\n--- {label} ---")
    print(f"  Status  : {job.status}")
    print(f"  Outcome : {job.outcome}")
    if job.runs:
        run = job.runs[0]
        result = client.get_results(run.id)
        print(f"  Backend : {result.backend_name if hasattr(result, 'backend_name') else 'N/A'}")
        print(f"  Shots   : {result.total_shots}")
        print(f"  Counts  : {dict(sorted(result.counts.items()))}")

## Fidelity-First vs Cost-Aware Scheduling

**Fidelity-First** selects the backend expected to produce the most accurate
results based on historical error rates and calibration data. This is the
right choice for research workloads where result quality matters most.

**Cost-Aware** considers execution cost alongside fidelity. For workloads
that can tolerate slightly lower fidelity (e.g., variational algorithm
iterations), cost-aware scheduling can significantly reduce cloud compute
expenses.

In both cases, QONTOS guarantees:
- Full **execution proofs** for integrity verification
- Consistent **result format** regardless of backend
- Automatic **circuit translation** to each backend's native gate set

In [ ]:
# Clean up
client.close()
print("Client closed.")

## Summary

In this notebook you learned:

- How to list available backends with `client.list_backends()`
- How to configure scheduling policies: `FIDELITY_FIRST`, `COST_AWARE`, etc.
- How to submit the same circuit under different policies and compare results
- The tradeoffs between fidelity-optimized and cost-optimized scheduling

**Next:** [05_vqe_h2.ipynb](05_vqe_h2.ipynb) applies these concepts to a
real quantum chemistry problem -- finding the ground state of H2.